# Figure 4 | Cortical LR communication landscape

Cleaned notebook for the public NVU AD spatial transcriptomics repository. Paths are repository-relative; set `NVU_PROJECT_ROOT` if running from another working directory.


In [ ]:
# Repository-relative paths
import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("NVU_PROJECT_ROOT", Path.cwd().parent)).resolve()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
REFERENCE_DIR = PROJECT_ROOT / "references"
FIGURE_DIR = RESULTS_DIR / "figure4"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc
from stereosite.plot.scii import ligrec
import anndata
import os
from stereosite.scii import intensities_count
import sys
import stereosite
import matplotlib.pyplot as plt

samples_list = [
  # "GSM8330060_B02009F6",
  # "GSM8330061_B02008D2",
  # "GSM8330062_C02248B5",
  "GSM8330063_A02092E1",
  "GSM8330064_B02008C6",
  "GSM8330065_B01806B6",
  "GSM8330066_D02175A4",
  "GSM8330067_B01809C2",
  "GSM8330068_B01809A4",
  "GSM8330069_B01809A3",
  "GSM8330070_D02175A6",
  "GSM8330071_B01806B5"
]

In [ ]:
data = sc.read('../data/h5ad/Cortex/Subtype/GSM8330067_B01809C2.NVU_subtype.h5ad')

In [ ]:

filepath = '../data/h5ad/Cortex/Subtype/'
interactiondb_file = "../references/stereosite/CellChatDB.human.expanded.csv"
resultpath = '../results/07_Stereosite/all_pairs-100'
distance_threshold = 150
desired_areas = ['L1','L23','L456','WM']

# 确保结果目录存在
os.makedirs(resultpath, exist_ok=True)

# 循环处理每个样本
for profile in samples_list:
    print(f"\n{'='*60}")
    print(f"Processing sample: {profile}")
    print(f"{'='*60}")
    
    # 构建数据文件路径
    data_file = os.path.join(filepath, profile + '.NVU_subtype.h5ad')
    
    # 检查文件是否存在
    if not os.path.exists(data_file):
        print(f"Warning: File not found for {profile}: {data_file}")
        print(f"Skipping {profile}...")
        continue
    
    try:
        # Load the data
        print(f"Loading data from: {data_file}")
        adata = anndata.read(data_file)
        # 脑区合并映射
        region_merge_map = {
            'L1': 'L1',
            'L2': 'L23',
            'L23': 'L23',
            'L3': 'L23',
            'L4': 'L456',
            'L45': 'L456',
            'L456': 'L456',
            'L5': 'L456',
            'L6': 'L456',
            'WM': 'WM'
        }

        adata.obs['area_m'] = adata.obs['area'].map(region_merge_map)


        adata.obs['celltype_unit'] = adata.obs['celltype_unit'].astype(str)

        # Neuron 用 celltype 替换
        mask = adata.obs['celltype_unit'] == 'Neuron'
        adata.obs.loc[mask, 'celltype_unit'] = adata.obs.loc[mask, 'celltype']

        # 再用关键词把细分神经元归并为 EX/IN
        EX_keywords = ['IT neurons', 'ET neurons', 'NP neurons', 'CAR3 neurons', 
                       'CT neurons', 'L6b neurons']
        IN_keywords = ['PVALB', 'VIP', 'SST', 'LAMP5', 'RELN']

        for idx, val in adata.obs['celltype_unit'].items():
            if any(kw in val for kw in EX_keywords):
                adata.obs.at[idx, 'celltype_unit'] = 'EX'
            elif any(kw in val for kw in IN_keywords):
                adata.obs.at[idx, 'celltype_unit'] = 'IN'

        adata.obs['celltype_unit'] = adata.obs['celltype_unit'].astype('category')

        adata = adata[adata.obs['dist'] == 1].copy()
        # Combine 'x' and 'y' into spatial coordinates
        spatial_coords = np.vstack((adata.obs['x'], adata.obs['y'])).T
        adata.obsm['spatial'] = spatial_coords
        
        # 循环处理每个区域
        for area in desired_areas:
            print(f"\n  Processing area: {area}")
            
            # 过滤数据
            filtered_data = adata[adata.obs['area_m'].isin([area])]
            
            # 检查过滤后的数据是否为空
            if filtered_data.n_obs == 0:
                print(f"    Warning: No data for area {area} in sample {profile}, skipping...")
                continue
            
            print(f"    Cells in area {area}: {filtered_data.n_obs}")
            
            # 执行 intensities_count
            print(f"    Computing cell-cell interactions...")
            # scii_dict = intensities_count(
            #     filtered_data, 
            #     interactiondb_file, 
            #     distance_threshold=distance_threshold, 
            #     anno='subcelltype', 
            #     jobs=4
            # )
            scii_dict = intensities_count(
                filtered_data, 
                interactiondb_file, 
                distance_threshold=distance_threshold, 
                anno='celltype_unit', 
                jobs=4,
                use_raw=False  # ← 加这个
            )
                        # 获取结果
            intensity_df = scii_dict['intensities']
            pvalues_df = scii_dict['pvalues']
            
            # 定义保存路径
            intensity_csv_path = os.path.join(resultpath, f"{profile}_{area}_intensity_df.csv")
            pvalues_csv_path = os.path.join(resultpath, f"{profile}_{area}_pvalues_df.csv")
            
            # 保存结果
            intensity_df.to_csv(intensity_csv_path, index=True)
            pvalues_df.to_csv(pvalues_csv_path, index=True)
            
            print(f"    Results saved:")
            print(f"      - {intensity_csv_path}")
            print(f"      - {pvalues_csv_path}")
        
        print(f"\nCompleted processing for sample: {profile}")
        
    except Exception as e:
        print(f"Error processing sample {profile}: {str(e)}")
        print(f"Continuing with next sample...")
        continue

print(f"\n{'='*60}")
print("All samples processed!")
print(f"{'='*60}")

In [ ]:
1